# Research Study: Evaluating Representation, Tagging, and Search/Retrieval in EURLEX Document Search
## Comparative Analysis of 3 Real-World Search Cases (9 Curated Benchmark Queries)

**Experiment Directory:** `experiments/retrieval_representation_study`  
**Date:** 24 October 2026  
**Evaluation Model:** `mistral-small-2603` (Mistral API)

### Real-World Search Archetypes Evaluated:
1. **Case 1: Singularity Match (Q1–Q3):** Narrow, exact target queries (e.g. specific hospital State aid case, specific spectrum decision).
2. **Case 2: Multi-Match (Q4–Q6):** Thematic regulatory regimes with $\ge 3$ ground-truth legislative acts (e.g. bilateral aviation safety, passenger car CO2 eco-innovations, Schengen acquis provisions).
3. **Case 3: Zero-Match (Q7–Q9):** Negative queries with 0 ground truth in corpus (emerging dual-use tech, US state statutes, non-existent regulations).


In [ ]:
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('intermediate_outputs')
FIG_DIR = Path('figures')

# Load benchmark metrics and evaluations
with open(DATA_DIR / 'benchmark_metrics.json') as f:
    metrics = json.load(f)

with open(DATA_DIR / 'query_evaluations.json') as f:
    evals = json.load(f)

with open(DATA_DIR / 'chunk_representation_audit.json') as f:
    chunk_audit = json.load(f)

with open(DATA_DIR / 'tag_utilization_audit.json') as f:
    tag_audit = json.load(f)

print('Loaded audit files successfully.')


## 1. Likelihood & Realism Analysis of the 3 Search Cases
In professional legal research across a 10,000+ legislative corpus:

| Search Case | Definition & Intent | Typical Corpus Prevalence | Real-World User Behavior |
| :--- | :--- | :--- | :--- |
| **Case 1: Singularity Match** | Specific case/dispute or individual narrow decision (e.g. `Brussels IRIS hospitals State aid`, `470-790 MHz frequency band`). | **~25% – 30%** of legal queries | Common when practitioners possess a known docket, dispute name, or CELEX citation. User expects **exactly 1 authoritative document at Rank 1**. |
| **Case 2: Multi-Match** | Framework regimes, evolving directives, and thematic regulations (e.g. `passenger cars CO2 emissions innovative technology`, `Schengen acquis participation provisions`). | **~60% – 65%** of legal queries | **The dominant search pattern in EU law.** EU secondary legislation operates through clusters of implementing decisions, amendments, and opt-ins. User expects **at least 3 to 10 relevant documents**, requiring multi-document synthesis. |
| **Case 3: Zero-Match** | Out-of-corpus, out-of-jurisdiction, or non-existent laws (e.g. `California consumer privacy act CCPA`, `quantum computing export controls`). | **~10% – 15%** of legal queries | Users test jurisdictional boundaries (US vs EU law) or inquire about post-cutoff emerging topics. Critical for testing system refusal vs hallucination. |


## 2. Corpus Representation Audit: The 92.75% Information Loss
Audit reveals that **99.98% of documents in the index consist of a single un-chunked document**, with chunk lengths reaching up to **213,675 tokens**.
Because `all-MiniLM-L6-v2` enforces a hard maximum sequence length of **256 tokens**, **92.75% of the entire legal corpus tokens are silently truncated** and discarded from vector representations.


In [ ]:
rep_summary = pd.DataFrame([{
    'Metric': 'Total Indexed Documents',
    'Value': f"{chunk_audit['total_documents']:,}"
}, {
    'Metric': 'Total Chunks in FAISS',
    'Value': f"{chunk_audit['total_chunks']:,}"
}, {
    'Metric': 'Single Chunk Documents (%)',
    'Value': f"{chunk_audit['chunks_per_doc_distribution']['pct_single_chunk_docs']:.2f}%"
}, {
    'Metric': 'Median Chunk Token Count',
    'Value': f"{chunk_audit['token_length_distribution']['median']:,}"
}, {
    'Metric': 'Max Chunk Token Count',
    'Value': f"{chunk_audit['token_length_distribution']['max']:,}"
}, {
    'Metric': 'Embedding Max Sequence Length',
    'Value': f"{chunk_audit['embedding_truncation_impact']['max_sequence_length']} tokens"
}, {
    'Metric': 'Corpus Tokens Discarded / Lost',
    'Value': f"{chunk_audit['embedding_truncation_impact']['corpus_token_loss_percentage']:.2f}%"
}, {
    'Metric': 'Preamble Boilerplate in First 256 Toks',
    'Value': f"{chunk_audit['preamble_bias_audit']['preamble_boilerplate_percentage']:.1f}%"
}])
rep_summary


### Figure 1: Corpus Truncation and Chunk Distribution


## 3. Tagging Quality & Utilization Audit
1. **EuroVoc URI Keyword Mismatch:** Raw EuroVoc strings from SPARQL scraping use URI format (`http://eurovoc.europa.eu/1252`), which fails substring keyword matching in `map_eurovoc_raw`.
2. **Category Collapse:** In TF-IDF prototype matching, generic terms like 'union' and 'budget' cause **92.2% of documents to be tagged EU-wide** and **82.0% tagged Financial Regulation**.
3. **Decoupled Search:** The retrieval engine gives tags **0.0% weight** in retrieval and ranking.


In [ ]:
tag_summary = pd.DataFrame([{
    'Dimension': 'Jurisdiction',
    'Dominant Category': 'EU-wide',
    'Count': tag_audit['category_imbalance_findings']['Jurisdiction']['count'],
    'Corpus %': f"{tag_audit['category_imbalance_findings']['Jurisdiction']['percentage_of_all_docs']:.1f}%",
    'Diagnosis': 'Extreme category collapse from generic stop-word keywords'
}, {
    'Dimension': 'Regulatory Topics',
    'Dominant Category': 'Financial Regulation',
    'Count': tag_audit['category_imbalance_findings']['Regulatory Topics']['count'],
    'Corpus %': f"{tag_audit['category_imbalance_findings']['Regulatory Topics']['percentage_of_all_docs']:.1f}%",
    'Diagnosis': 'Over-triggering across diverse non-financial legislation'
}, {
    'Dimension': 'Search Engine Utilization',
    'Dominant Category': 'All Dimensions',
    'Count': 0,
    'Corpus %': '0.0%',
    'Diagnosis': 'Tags are 100% ignored during FAISS search and BM25 re-ranking'
}])
tag_summary


### Figure 2: Tag Collapse & Imbalance


## 4. Evaluation of 9 Benchmark Queries across 3 Search Cases
Evaluating search retrieval precision, multi-target recovery in Top 5, false positive rates, and Mistral `mistral-small-2603` summaries.


In [ ]:
eval_rows = []
for e in evals:
    eval_rows.append({
        'ID': e['query_id'],
        'Case': e['case_type'].replace('_', ' ').title(),
        'Query': e['query'],
        'Top CELEX': e['top_celex'],
        'Peak Score': round(e['top_score'], 3),
        'GT Total': e['ground_truth_total'],
        'GT in Top 5': e['ground_truth_found_in_top5'],
        'P@1': e['precision_at_1'],
        'P@3': e['precision_at_3'],
        'P@5': e['precision_at_5'],
        'False Pos?': 'YES' if e['false_positive_returned'] else 'NO',
        'Tag Weight': '0.0%',
        'Mistral Status': e['summariser_status'],
        'Tokens': e['tokens_consumed']
    })
df_evals = pd.DataFrame(eval_rows)
df_evals


### Figure 3: Score Calibration across the 3 Search Cases


### Figure 4: Retrieval Performance Metrics


## 5. Summariser Fidelity Analysis (mistral-small-2603)
Key Findings on LLM behavior:
- **Mistral-small-2603 is highly truthful and attentive:** For all 3 negative queries (Q7, Q8, Q9), Mistral explicitly noted in its Executive Summary that the retrieved documents do not directly address the query topic (e.g. noting that CCPA is a California statute, or that quantum computing export controls are absent).
- **Prompt Forced Hallucination Risk:** Because the system prompt rigidly demanded cross-document synthesis across ALL 5 retrieved documents without a fallback refusal condition, Mistral was forced to draw speculative analogies (e.g. linking Chinese solar panel anti-dumping undertakings to quantum computing, or UNCLOS fisheries treaties to asteroid mining).
- **Remedy:** Introduce a confidence threshold in the search engine (e.g. top_score < 0.40 returns 'No relevant legislation found') and update the prompt to allow graceful refusal.


In [ ]:
with open(DATA_DIR / 'benchmark_results_raw.json') as f:
    raw_data = json.load(f)

for r in raw_data['raw_results'][6:]:
    print('='*70)
    print(f"{r['meta']['id']} ({r['meta']['query']})")
    print('Mistral Self-Correction / Mismatch Flag:')
    first_para = r['summariser_result']['summary'].split('##')[1] if '##' in r['summariser_result']['summary'] else r['summariser_result']['summary'][:300]
    print(first_para.strip()[:400] + '...')
